# HybridLLM — Generalized to N=5 Models

**Paper:** Ding et al., *Hybrid LLM: Cost-Efficient and Quality-Aware Query Routing*, ICLR 2024 (arXiv:2404.14618)

The paper's router is binary (small vs. large). This notebook **generalizes it to 5 models**: instead
of a binary classifier, the router becomes a **regression model** that predicts each of the 5
models' expected quality for a query, directly from the query text. At decision time, we pick the
**cheapest model whose predicted quality is within a margin `t` of the best predicted quality** —
sweeping `t` reproduces the same cost-vs-quality tradeoff curve as the original paper's θ sweep,
just generalized to N models instead of 2.

This also removes the need for the paper's 10-sample probabilistic labels (`r_prob`/`r_trans`):
those existed specifically to fix a *binary* label collapsing to all-0/all-1 when the quality gap
is large. A regression target doesn't have that failure mode, so 1 deterministic sample per model
is enough — net effect, more models but fewer total generations than the original 2-model setup.

Results are saved to `hybrid_llm_results.json` for later comparison against the RL/multi-agent
notebook (same 5 models, same 15,000-query pool, same BARTScore quality metric).

In [ ]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs], check=True)

pip_install("transformers", "accelerate", "datasets", "sentencepiece", "scikit-learn", "protobuf")
print("installed.")


In [ ]:
import os, math, json, random, pickle, shutil, time

# Pin this notebook to GPU 0. (GPU 1 on this shared machine is already heavily used by another
# process -- checked via nvidia-smi -- so both notebooks run on GPU 0, one at a time: finish one,
# Kernel > Shut Down Kernel to free its GPU memory, then run the other.)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM,
    AutoModel, BartTokenizer, BartForConditionalGeneration,
)

SEED = 0
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
torch.set_default_dtype(torch.float32)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, " (physical GPU 0, per CUDA_VISIBLE_DEVICES)")
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} | total memory: {gpu_mem_gb:.1f} GB")

# --- Local storage, scoped to one project folder (not system-wide) ---
# Model weights go to the DEFAULT Hugging Face cache (~/.cache/huggingface), on purpose:
# that's shared across both notebooks AND persists across your future months of use --
# nothing here gets re-downloaded next time. Only this project's own checkpoints/results
# live in WORK_DIR, so cleanup later is a single `rm -rf` of one small folder.
WORK_DIR = os.path.expanduser("~/hybridllm_project")
CKPT_DIR = os.path.join(WORK_DIR, "ckpt_shared")   # SAME path in both notebooks -> shared checkpoints
OUTPUT_DIR = os.path.join(WORK_DIR, "outputs")
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Work dir:", WORK_DIR)

def save_ckpt(name, obj):
    with open(f"{CKPT_DIR}/{name}.pkl", "wb") as f:
        pickle.dump(obj, f)

def load_ckpt(name):
    path = f"{CKPT_DIR}/{name}.pkl"
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    return None

def load_with_retry(model_class, model_id, **kwargs):
    """Retries with a clean re-download if a cached safetensors file is corrupted."""
    try:
        return model_class.from_pretrained(model_id, **kwargs)
    except Exception as e:
        print(f"  [{model_id}] initial load failed ({type(e).__name__}: {e}).")
        print(f"  [{model_id}] clearing cached snapshot and re-downloading cleanly...")
        cache_root = os.path.join(os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface")), "hub")
        repo_folder = "models--" + model_id.replace("/", "--")
        target = os.path.join(cache_root, repo_folder)
        if os.path.exists(target):
            shutil.rmtree(target)
            print(f"  removed {target}")
        return model_class.from_pretrained(model_id, force_download=True, **kwargs)


## Config

`N_TRAIN`/`N_TEST` are the real target scale (10,000 / 5,000). Run the timing probe (later in this
notebook) BEFORE launching the full label-building loop so you know what you're committing to.
Everything expensive below is checkpointed in small chunks, so an interruption only costs you that
chunk -- just re-run the same cell and it resumes.

**Hardware note:** with 2x L40S 48GB, Llama-2-7b-chat fits comfortably in full fp16 (~14GB) -- no
4-bit quantization needed (that was purely a Kaggle 16GB workaround). All 5 models + BARTScorer
together need roughly 25-28GB, well inside one 48GB card.

In [ ]:
N_TRAIN = 10000
N_TEST = 5000
MAX_NEW_TOKENS = 64
CHUNK_SIZE = 100            # queries per checkpoint during label-building / RL loop
ROUTER_EPOCHS = 6
ROUTER_LR = 2e-5
ROUTER_BATCH = 16

MODEL_IDS = [
    ("flan-t5-small",   "google/flan-t5-small",            False, 80e6),
    ("flan-t5-base",    "google/flan-t5-base",             False, 250e6),
    ("flan-t5-large",   "google/flan-t5-large",            False, 780e6),
    ("flan-t5-xl",      "google/flan-t5-xl",               False, 3e9),
    ("llama-2-7b-chat", "NousResearch/Llama-2-7b-chat-hf", True,  7e9),
]
MAX_PARAMS = max(p for _, _, _, p in MODEL_IDS)
LARGEST_MODEL_NAME = max(MODEL_IDS, key=lambda m: m[3])[0]
print("Largest model (baseline / cost=1.0):", LARGEST_MODEL_NAME)


## Data — MixInstruct

Same seed + same sampling code in both notebooks -> identical 15,000 queries in both. HybridLLM
splits them 10k/5k; the RL notebook recombines them into one 15k stream (no train/test split there,
per the design).

In [ ]:
cached = load_ckpt("examples")
if cached:
    train_examples, test_examples = cached
    print(f"loaded from checkpoint: train={len(train_examples)}  test={len(test_examples)}")
else:
    print("=== Loading MixInstruct ===")
    ds_train = load_dataset("llm-blender/mix-instruct", split="train")
    ds_test = load_dataset("llm-blender/mix-instruct", split="validation")

    def build_query(ex):
        instr = (ex.get("instruction") or "").strip()
        inp = (ex.get("input") or "").strip()
        return (instr + "\n" + inp).strip() if instr else inp

    def sample_examples(ds, n, seed_offset=0):
        rng = random.Random(SEED + seed_offset)
        idx = rng.sample(range(len(ds)), min(n, len(ds)))
        out = []
        for i in idx:
            ex = ds[i]
            q = build_query(ex)
            ref = (ex.get("output") or "").strip()
            if q and ref:
                out.append({"query": q, "reference": ref})
        return out

    train_examples = sample_examples(ds_train, N_TRAIN, seed_offset=1)
    test_examples = sample_examples(ds_test, N_TEST, seed_offset=2)
    save_ckpt("examples", (train_examples, test_examples))
    print(f"train={len(train_examples)}  test={len(test_examples)}")


## Load the 5 shared models

Same 5 models in both notebooks -> apples-to-apples comparison. Staying in the FLAN-T5 family for
the small end (same architecture, just scaled) plus Llama-2-7b-chat at the top gives a wide, clean
cost gradient without downloading unrelated large models. All loaded in plain fp16, no quantization
needed on this hardware.

In [ ]:
MODEL_REGISTRY = []
for name, model_id, is_causal, params in MODEL_IDS:
    print(f"loading {name} ({model_id}) ...")
    tok = AutoTokenizer.from_pretrained(model_id)
    if is_causal:
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        tok.padding_side = "left"
        model = load_with_retry(
            AutoModelForCausalLM, model_id,
            torch_dtype=torch.float16, device_map="auto",
        )
    else:
        model = load_with_retry(
            AutoModelForSeq2SeqLM, model_id, torch_dtype=torch.float16
        ).to(DEVICE)
    model.eval()
    MODEL_REGISTRY.append({"name": name, "tok": tok, "model": model, "causal": is_causal, "params": params})
    print("  done.")

for m in MODEL_REGISTRY:
    m["cost"] = m["params"] / MAX_PARAMS
    print(f"{m['name']:>16}: params={m['params']:.2e}  normalized cost={m['cost']:.4f}")

N_ARMS = len(MODEL_REGISTRY)
MODEL_NAMES = [m["name"] for m in MODEL_REGISTRY]
LARGEST_IDX = MODEL_NAMES.index(LARGEST_MODEL_NAME)

def generate_one(entry, query, max_new_tokens=MAX_NEW_TOKENS):
    tok, model = entry["tok"], entry["model"]
    enc = tok(query, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
    with torch.no_grad():
        if entry["causal"]:
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
            text = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
        else:
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
            text = tok.decode(out[0], skip_special_tokens=True)
    return text

print("\nagents ready.")


## Timing probe — run this before launching the full label-building / RL loop

Times one generation per model, then projects total runtime for the actual query counts you're
about to commit to. On L40S this should be noticeably faster than the Kaggle T4 estimate was.

In [ ]:
probe_query = train_examples[0]["query"]
per_model_time = {}
for m in MODEL_REGISTRY:
    t0 = time.time()
    _ = generate_one(m, probe_query)
    per_model_time[m["name"]] = time.time() - t0
    print(f"{m['name']:>16}: {per_model_time[m['name']]:.2f}s")

total_per_query_all_models = sum(per_model_time.values())
print(f"\nsum of all 5 models, one generation each: {total_per_query_all_models:.2f}s")
print(f"estimated label-building time, {len(train_examples)} train queries x 5 models: "
      f"{total_per_query_all_models * len(train_examples) / 3600:.1f} hours")
print(f"estimated label-building time, {len(test_examples)} test queries x 5 models: "
      f"{total_per_query_all_models * len(test_examples) / 3600:.1f} hours")


## BARTScore — quality metric (same as before)

In [ ]:
BATCH_SIZE_SCORE = 8
BART_CKPT = "facebook/bart-large-cnn"

class BARTScorer:
    def __init__(self, checkpoint=BART_CKPT, device=DEVICE, max_length=1024):
        self.device = device
        self.max_length = max_length
        self.tokenizer = BartTokenizer.from_pretrained(checkpoint)
        self.model = load_with_retry(BartForConditionalGeneration, checkpoint).to(device)
        self.model.eval()
        self.loss_fct = nn.NLLLoss(reduction="none", ignore_index=self.model.config.pad_token_id)
        self.lsm = nn.LogSoftmax(dim=1)

    @torch.no_grad()
    def score(self, srcs, tgts, batch_size=8):
        scores = []
        for i in range(0, len(srcs), batch_size):
            src_batch = srcs[i:i+batch_size]
            tgt_batch = tgts[i:i+batch_size]
            enc_src = self.tokenizer(src_batch, max_length=self.max_length, truncation=True,
                                      padding=True, return_tensors="pt").to(self.device)
            enc_tgt = self.tokenizer(tgt_batch, max_length=self.max_length, truncation=True,
                                      padding=True, return_tensors="pt").to(self.device)
            tgt_ids = enc_tgt["input_ids"]
            output = self.model(
                input_ids=enc_src["input_ids"], attention_mask=enc_src["attention_mask"],
                labels=tgt_ids,
            )
            logits = output.logits.view(-1, self.model.config.vocab_size)
            loss = self.loss_fct(self.lsm(logits), tgt_ids.view(-1))
            loss = loss.view(tgt_ids.shape[0], -1)
            avg_loss = loss.sum(dim=1) / (tgt_ids != self.model.config.pad_token_id).sum(dim=1)
            scores.extend((-avg_loss).tolist())
        return scores

bart_scorer = BARTScorer()

def quality(hypotheses, references):
    """q(hypothesis) := BARTScore(reference -> hypothesis)."""
    return bart_scorer.score(references, hypotheses, batch_size=BATCH_SIZE_SCORE)

print("BARTScorer ready.")


## Label-building: query every model, on train AND test

Every query needs a quality score from **all 5 models** — on train, these become the router's
regression targets; on test, these let us evaluate the full `t` sweep (what quality *would* result
from any routing choice), same as the original notebook precomputed `q_s_test`/`q_l_test` for both
candidate models.

Chunked and checkpointed: if interrupted, re-running this cell resumes from the last saved chunk
instead of starting over.

In [ ]:
def run_labeling(examples, ckpt_name, chunk_size=CHUNK_SIZE):
    results = load_ckpt(ckpt_name) or []
    n = len(examples)
    start = len(results)
    if start >= n:
        print(f"{ckpt_name}: already complete ({n} rows)")
        return results
    print(f"{ckpt_name}: resuming from {start}/{n}")
    t_start = time.time()
    for chunk_start in range(start, n, chunk_size):
        chunk = examples[chunk_start: chunk_start + chunk_size]
        queries = [e["query"] for e in chunk]
        refs = [e["reference"] for e in chunk]
        per_model_quality = {}
        for m in MODEL_REGISTRY:
            gens = [generate_one(m, q) for q in queries]
            scores = quality(gens, refs)
            per_model_quality[m["name"]] = scores
        for i in range(len(chunk)):
            row = {m["name"]: per_model_quality[m["name"]][i] for m in MODEL_REGISTRY}
            results.append(row)
        save_ckpt(ckpt_name, results)
        elapsed = time.time() - t_start
        done = len(results) - start
        rate = elapsed / max(done, 1)
        remaining = (n - len(results)) * rate
        print(f"  [{ckpt_name}] {len(results)}/{n} ({100*len(results)/n:.1f}%) -- "
              f"est. remaining: {remaining/3600:.2f}h")
    return results

print("=== Labeling TRAIN set ===")
train_quality_rows = run_labeling(train_examples, "train_quality_rows")

print("\n=== Labeling TEST set ===")
test_quality_rows = run_labeling(test_examples, "test_quality_rows")

Y_train = np.array([[row[name] for name in MODEL_NAMES] for row in train_quality_rows])  # [N_TRAIN, 5]
Y_test  = np.array([[row[name] for name in MODEL_NAMES] for row in test_quality_rows])   # [N_TEST, 5]
print("\nY_train shape:", Y_train.shape, " Y_test shape:", Y_test.shape)


## Router model — DeBERTa-v3-small backbone, 5-output regression head

Input: query text. Output: 5 predicted quality scores (one per model), learned via MSE regression
against the real BARTScores from the label-building step above. At serving time, the router needs
**no model calls at all** to make its routing decision -- the whole point of training it.

In [ ]:
ROUTER_ID = "microsoft/deberta-v3-small"

# Earlier float16 loads can leave torch's global default dtype at float16, which would silently
# load this backbone in fp16 too -> dtype mismatch against the fp32 head. Force fp32 explicitly.
torch.set_default_dtype(torch.float32)

router_tok = AutoTokenizer.from_pretrained(ROUTER_ID)
router_backbone = AutoModel.from_pretrained(ROUTER_ID, torch_dtype=torch.float32).to(DEVICE)

class RegressionRouterHead(nn.Module):
    """State (query embedding) -> predicted quality for each of the N models."""
    def __init__(self, hidden_size, n_models):
        super().__init__()
        self.linear = nn.Linear(hidden_size, n_models)
    def forward(self, pooled):
        return self.linear(pooled)

router_head = RegressionRouterHead(router_backbone.config.hidden_size, N_ARMS).to(DEVICE).float()

def encode_queries(queries, max_length=128):
    return router_tok(queries, return_tensors="pt", padding=True, truncation=True, max_length=max_length)

def router_forward(queries):
    enc = encode_queries(queries)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    out = router_backbone(**enc)
    pooled = out.last_hidden_state[:, 0]
    return router_head(pooled)   # [batch, N_ARMS] predicted quality per model

print("router model ready.")


## Train the router (regression, MSE loss)

In [ ]:
train_queries = [e["query"] for e in train_examples]
Y_train_t = torch.tensor(Y_train, dtype=torch.float32)

opt = torch.optim.AdamW(list(router_backbone.parameters()) + list(router_head.parameters()), lr=ROUTER_LR)

n = len(train_queries)
for epoch in range(ROUTER_EPOCHS):
    perm = np.random.permutation(n)
    epoch_loss = 0.0
    for start in range(0, n, ROUTER_BATCH):
        batch_idx = perm[start:start + ROUTER_BATCH]
        batch_q = [train_queries[j] for j in batch_idx]
        batch_y = Y_train_t[batch_idx].to(DEVICE)
        preds = router_forward(batch_q)
        loss = F.mse_loss(preds, batch_y)
        opt.zero_grad(); loss.backward(); opt.step()
        epoch_loss += loss.item() * len(batch_idx)
    print(f"epoch {epoch+1}/{ROUTER_EPOCHS}  mse={epoch_loss/n:.4f}")

print("router trained.")


## Evaluate — sweep margin `t`, generalized cost-advantage vs. quality-drop curve

For each test query: get the router's 5 predicted quality scores (no model calls). Find the best
predicted score, then pick the **cheapest model whose predicted quality is within `t` of the best**.
Realized quality = that chosen model's REAL (precomputed) quality on that query. Sweeping `t` from 0
(always pick the model the router predicts is best, usually the largest) to large (always pick the
cheapest model) traces the full cost/quality tradeoff, same spirit as the original paper's θ sweep.

In [ ]:
@torch.no_grad()
def predict_test_quality(queries, batch_size=32):
    preds = []
    for start in range(0, len(queries), batch_size):
        batch = queries[start:start + batch_size]
        p = router_forward(batch).cpu().numpy()
        preds.append(p)
    return np.concatenate(preds, axis=0)

test_queries = [e["query"] for e in test_examples]
Y_pred_test = predict_test_quality(test_queries)   # [N_TEST, 5]

costs = np.array([m["cost"] for m in MODEL_REGISTRY])          # normalized, length 5
order_by_cost = np.argsort(costs)                                # cheapest first

baseline_quality = float(Y_test[:, LARGEST_IDX].mean())
print(f"Always-largest-model ({LARGEST_MODEL_NAME}) baseline quality: {baseline_quality:.4f}")

max_gap = float(np.percentile(Y_pred_test.max(axis=1) - Y_pred_test.min(axis=1), 95)) or 1.0
t_values = np.linspace(0.0, max_gap * 1.2, 21)

sweep = []
for t in t_values:
    chosen_idx = np.zeros(len(test_queries), dtype=int)
    for i in range(len(test_queries)):
        best_pred = Y_pred_test[i].max()
        acceptable = [j for j in order_by_cost if Y_pred_test[i, j] >= best_pred - t]
        chosen_idx[i] = acceptable[0] if acceptable else int(np.argmax(Y_pred_test[i]))

    realized_quality = Y_test[np.arange(len(test_queries)), chosen_idx]
    mean_quality = float(realized_quality.mean())
    mean_norm_cost = float(costs[chosen_idx].mean())
    cost_advantage_pct = 100.0 * (1.0 - mean_norm_cost)   # % cost saved vs. always using the largest model
    quality_drop_pct = 100.0 * (baseline_quality - mean_quality) / abs(baseline_quality)

    sweep.append({
        "margin_t": float(t),
        "cost_advantage_pct": cost_advantage_pct,
        "mean_quality": mean_quality,
        "mean_norm_cost": mean_norm_cost,
        "quality_drop_pct": quality_drop_pct,
        "model_distribution": dict(Counter([MODEL_NAMES[j] for j in chosen_idx])),
    })
    print(f"t={t:5.3f} | cost_adv={cost_advantage_pct:6.1f}% | quality={mean_quality:8.4f} | "
          f"drop={quality_drop_pct:+6.2f}%")


## Save results + raw (parameter-count) cost totals, for comparison against RL

In [ ]:
best_row = min(sweep, key=lambda r: abs(r["quality_drop_pct"]))   # closest-to-zero quality drop
print(f"Best 'near-free' operating point: t={best_row['margin_t']:.3f} -> "
      f"{best_row['cost_advantage_pct']:.1f}% cost savings, {best_row['quality_drop_pct']:+.2f}% quality change")

PARAMS = {m["name"]: m["params"] for m in MODEL_REGISTRY}

# Raw parameter-count cost totals (headline comparison unit against the RL notebook)
label_building_cost_raw = len(train_examples) * sum(PARAMS.values())        # every model called on every train query
test_deployment_cost_raw = sum(PARAMS[MODEL_NAMES[j]] for j in
                                [order_by_cost[0] if False else None])       # placeholder, recomputed below

# recompute test deployment raw cost properly using the chosen model distribution at best_row's t
best_t = best_row["margin_t"]
chosen_idx_best = np.zeros(len(test_queries), dtype=int)
for i in range(len(test_queries)):
    best_pred = Y_pred_test[i].max()
    acceptable = [j for j in order_by_cost if Y_pred_test[i, j] >= best_pred - best_t]
    chosen_idx_best[i] = acceptable[0] if acceptable else int(np.argmax(Y_pred_test[i]))
test_deployment_cost_raw = float(sum(PARAMS[MODEL_NAMES[j]] for j in chosen_idx_best))

total_cost_raw = label_building_cost_raw + test_deployment_cost_raw

results = {
    "system": "hybridllm_n5_regression_router",
    "n_train": len(train_examples),
    "n_test": len(test_examples),
    "model_names": MODEL_NAMES,
    "model_params": PARAMS,
    "max_params": MAX_PARAMS,
    "baseline_model": LARGEST_MODEL_NAME,
    "baseline_quality": baseline_quality,
    "sweep": sweep,
    "best_operating_point": best_row,
    "label_building_cost_raw_params": label_building_cost_raw,
    "test_deployment_cost_raw_params_at_best_t": test_deployment_cost_raw,
    "total_cost_raw_params": total_cost_raw,
}

with open(os.path.join(OUTPUT_DIR, "hybrid_llm_results.json"), "w") as f:
    json.dump(results, f, indent=2)

print(f"\nlabel-building cost (raw params):        {label_building_cost_raw:,.0f}")
print(f"test deployment cost at best t (raw params): {test_deployment_cost_raw:,.0f}")
print(f"TOTAL HybridLLM cost (raw params):        {total_cost_raw:,.0f}")
print("saved " + os.path.join(OUTPUT_DIR, "hybrid_llm_results.json"))
